# Corrected BanglaBERT run — corrected evaluation

Select a T4 GPU runtime, then run each cell in order. Upload `banglabert-evaluation-inputs.zip` when prompted. This bundle uses the corrected local code and frozen 149-item paired population. No image or article-body download is needed for this text-only run. Historical neural scores are not reused.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
import zipfile
uploaded = files.upload()
assert "banglabert-evaluation-inputs.zip" in uploaded
with zipfile.ZipFile("banglabert-evaluation-inputs.zip") as archive:
    archive.extractall("/content/bmpb-evaluation")
%cd /content/bmpb-evaluation


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


In [ ]:
import hashlib, json
from pathlib import Path
import pandas as pd
from bmpb.crossval import make_folds, cross_validate
manifest = json.loads(Path("run-manifest.json").read_text())
payload = Path("data/processed/corpus.csv").read_bytes()
assert hashlib.sha256(payload).hexdigest() == manifest["corpus_sha256"]
paired = pd.read_csv("data/processed/corpus.csv")
assert len(paired) == 149
assert [paired.iloc[f].item_id.tolist() for f in make_folds(paired)] == manifest["fold_item_ids"]
print("Verified frozen population and outer folds")


In [ ]:
result = cross_validate("configs/text/banglabert.yaml", corpus=paired, out="experiments/evaluation/paired_headlines")
print(json.dumps(result, indent=2))


In [ ]:
import shutil
shutil.copy("run-manifest.json", "experiments/run-manifest.json")
shutil.make_archive("banglabert-evaluation-results", "zip", "experiments")
files.download("banglabert-evaluation-results.zip")
